# Signal or Noise in Multi-Agent LLM-based Stock Recommendations?

Authors: George Fatouros, Kostas Metaxas
Published: 2026-04-19
Arxiv: https://arxiv.org/abs/2604.17327

## Abstract
We present the first portfolio-level validation of MarketSenseAI, a deployed multi-agent LLM equity system. All signals are generated live at each observation date, eliminating look-ahead bias. The system routes four specialist agents (News, Fundamentals, Dynamics, and Macro) through a synthesis agent that issues a monthly equity thesis and recommendation for each stock in its coverage universe, and we ask two questions: do its buy recommendations add value over both passive benchmarks and random selection, and what does the internal agent structure reveal about the source of the edge? On the S&P 500 cohort (19 months) the strong-buy equal-weight portfolio earns +2.18%/month against a passive equal-weight benchmark of +1.15% (approximating RSP), a +25.2% compound excess, and ranks at the 99.7th percentile of 10,000 Monte Carlo portfolios (p=0.003). The S&P 100 cohort (35 months) delivers a +30.5% compound excess over EQWL with consistent direction but formal significance not reached, limited by the small average selection of ~10 stocks per month. Non-negative least-squares projection of thesis embeddings onto agent embeddings reveals an adaptive-integration mechanism. Agent contributions rotate with market regime (Fundamentals leads on S&P 500, Macro on S&P 100, Dynamics acts as an episodic momentum signal) and this agent rotation moves in lockstep with both the sector composition of strong-buy selections and identifiable macro-calendar events, three independent views of the same underlying adaptation. The recommendation's cross-sectional Information Coefficient is statistically significant on S&P 500 (ICIR=+0.489, p=0.024). These results suggest that multi-agent LLM equity systems can identify sources of alpha beyond what classical factor models capture, and that the buy signal functions as an effective universe-filter that can sit upstream of any portfolio-construction process.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'FB', 'TSLA', 'BRK-B', 'JNJ', 'V', 'PG']
REBAL_FREQ = 'M'  # Monthly rebalancing
START_DATE = '2020-01-01'
END_DATE = '2023-01-01'

# Hypothesis
# The hypothesis is that the strong-buy recommendations from the multi-agent LLM system will outperform a passive equal-weight benchmark.

## Phase 2 — Data Download & Feature Computation

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download data
data = yf.download(UNIVERSE, start=START_DATE, end=END_DATE, group_by='ticker')

# Compute features
prices = data['Adj Close']
returns = prices.pct_change().dropna()

# Cross-sectional normalization
normalized_returns = (returns - returns.mean(axis=1)) / returns.std(axis=1)

## Phase 3 — Signal Generation & Portfolio Construction

In [ ]:
# Signal generation
signals = normalized_returns.rank(axis=1, pct=True).gt(0.9)

# Position sizing
weights = signals.where(signals, 0).div(signals.sum(axis=1), axis=0)

# Portfolio construction
portfolio_returns = (weights * returns).sum(axis=1)

## Phase 4 — Vectorized Backtest

In [ ]:
# Shift signals forward by 1 period to avoid look-ahead bias
signals_shifted = signals.shift(1)

# Recalculate portfolio returns with shifted signals
weights_shifted = signals_shifted.where(signals_shifted, 0).div(signals_shifted.sum(axis=1), axis=0)
portfolio_returns_shifted = (weights_shifted * returns).sum(axis=1)

## Phase 5 — Performance Metrics

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import norm

# Performance metrics
cumulative_returns = (1 + portfolio_returns_shifted).cumprod()
benchmark_returns = prices.mean(axis=1).pct_change().dropna()
benchmark_cumulative_returns = (1 + benchmark_returns).cumprod()

# Sharpe ratio
sharpe_ratio = np.mean(portfolio_returns_shifted) / np.std(portfolio_returns_shifted)

# Sortino ratio
downside_returns = portfolio_returns_shifted[portfolio_returns_shifted < 0]
sortino_ratio = np.mean(portfolio_returns_shifted) / np.std(downside_returns)

# Calmar ratio
max_drawdown = (cumulative_returns / cumulative_returns.cummax() - 1).min()
calmar_ratio = np.mean(portfolio_returns_shifted) / abs(max_drawdown)

# Plot equity curve
plt.figure(figsize=(10, 5))
plt.plot(cumulative_returns, label='Strategy')
plt.plot(benchmark_cumulative_returns, label='Benchmark')
plt.legend()
plt.title('Equity Curve')
plt.show()

print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown:.2f}')

## Phase 6 — Monitoring Stub

In [ ]:
# Monitoring stub
def monitor(live_data):
    live_prices = live_data['Adj Close']
    live_returns = live_prices.pct_change().dropna()
    live_signals = (live_returns > 0).shift(1)
    live_weights = live_signals.where(live_signals, 0).div(live_signals.sum(axis=1), axis=0)
    live_portfolio_returns = (live_weights * live_returns).sum(axis=1)
    print(f'Daily P&L: {live_portfolio_returns.iloc[-1]:.2f}')
    print(f'Current Positions: {live_weights.iloc[-1].where(live_weights.iloc[-1] > 0).dropna().index.tolist()}')

# Example usage
monitor(data)